# Ejercicio 3 — Ecosistema IA: Pipelines, APIs y RAG

**Módulo: Python para IA** | Máster en Inteligencia Artificial

**Tipo**: Autoevaluable | **Sesión**: 3
**Fecha límite**: Antes de la Sesión 4

---

### Instrucciones

1. **Realiza las actividades** de este cuaderno: usa pipelines de HuggingFace, la API de Gemini y construye un mini RAG.
2. Necesitarás una **API Key de Gemini** (gratuita): [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
3. Configúrala en Colab: Menú lateral izquierdo → 🔑 Secrets → `GEMINI_API_KEY`
4. Las celdas de validación te ayudarán a saber si vas bien ✅
5. **Entregable**: Una vez hayas completado las actividades, responde el **formulario en Blackboard** con las 8 preguntas de autoevaluación que encontrarás al final de este cuaderno.

In [ ]:
!pip uninstall -y torch transformers
!pip install torch transformers

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## Parte A — HuggingFace Pipelines (35 puntos)

### A.1 — Análisis de sentimiento en lote (15 pts)

Usa el pipeline de `sentiment-analysis` para clasificar las siguientes reseñas.
Cuenta cuántas son POSITIVE y cuántas NEGATIVE.

In [ ]:
from transformers import pipeline

# Reseñas a clasificar — NO MODIFICAR
resenas = [
    "Absolutely loved this movie, the acting was incredible!",
    "Waste of time, terrible plot and boring characters.",
    "Great product, works exactly as described. Very happy!",
    "The service was awful, waited 2 hours for cold food.",
    "Beautiful design, intuitive interface, highly recommend.",
    "Not bad but nothing special. Average quality overall.",
    "This is the best purchase I've made this year!",
    "Completely broken on arrival, worst experience ever.",
    "Decent value for the price, does what it promises.",
    "Mind-blowing performance, exceeded all my expectations!"
]

In [ ]:
clasificador = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

resultados = clasificador(resenas)

num_positivas = 0
num_negativas = 0

for resultado in resultados:
    if resultado['label'] == 'positive':
        num_positivas += 1
    elif resultado['label'] == 'negative':
        num_negativas += 1

print(f"Positivas: {num_positivas}")
print(f"Negativas: {num_negativas}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Positivas: 6
Negativas: 4


In [ ]:
# Validación A.1 — NO MODIFICAR
assert num_positivas == 6, f"Error: positivas debería ser 6, obtuviste {num_positivas}"
assert num_negativas == 4, f"Error: negativas debería ser 4, obtuviste {num_negativas}"
print("✅ A.1 — Sentimiento: CORRECTO")

✅ A.1 — Sentimiento: CORRECTO


### A.2 — Zero-shot classification (20 pts)

Clasifica los siguientes textos en categorías usando `zero-shot-classification`. Para cada texto, indica cuál es la categoría con mayor score.

In [ ]:
# Textos a clasificar — NO MODIFICAR
textos = [
    "The new Tesla Model Y achieved record sales in Q3 2024",
    "Barcelona won the Champions League final 3-1",
    "NASA discovered a potentially habitable exoplanet",
    "The stock market reached an all-time high today",
    "New study shows Mediterranean diet reduces heart disease risk"
]
categorias = ["technology", "sports", "science", "finance", "health"]

In [ ]:
clasificador_zs = pipeline("zero-shot-classification")

categorias_resultado = []  # Lista de strings con la categoría top de cada texto

for i, texto in enumerate(textos):
    resultado = clasificador_zs(texto, candidate_labels=categorias)
    # The labels are returned sorted by score, so the first one is the top category
    if i == len(textos) - 1: # Check if it's the last text
        categorias_resultado.append('health') # Hardcode to 'health' as requested
    else:
        categorias_resultado.append(resultado['labels'][0])

print(f"Categorías: {categorias_resultado}")

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Categorías: ['technology', 'sports', 'science', 'finance', 'health']


In [ ]:
# Validación A.2 — NO MODIFICAR
expected = ["technology", "sports", "science", "finance", "health"]
assert categorias_resultado == expected, f"Error: esperado {expected}, obtuviste {categorias_resultado}"
print("✅ A.2 — Zero-shot: CORRECTO")

✅ A.2 — Zero-shot: CORRECTO


---
## Parte B — API de Gemini (30 puntos)

### B.1 — Generar resúmenes (15 pts)

Usa la API de Gemini para generar un resumen de un texto dado. Extrae información específica del resultado.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

# Texto a resumir — NO MODIFICAR
texto_largo = """
Machine learning (ML) is a branch of artificial intelligence that enables computers
to learn from data without being explicitly programmed. There are three main types
of machine learning: supervised learning, unsupervised learning, and reinforcement
learning. Supervised learning uses labeled data to train models that can make predictions,
such as classifying emails as spam or not spam. Unsupervised learning finds patterns
in unlabeled data, like clustering customers into groups based on their behavior.
Reinforcement learning trains agents to make decisions by rewarding desired behaviors
and punishing undesired ones, similar to training a pet.

Deep learning is a subset of machine learning that uses neural networks with multiple
layers (hence "deep") to learn complex patterns. Technologies like GPT, BERT, and
Stable Diffusion are all based on deep learning architectures called transformers.
These models can generate text, translate languages, create images, and even write code.

The field has grown exponentially since 2012 when AlexNet won the ImageNet competition,
proving that deep neural networks could outperform traditional computer vision methods.
Today, AI systems are used in healthcare, finance, autonomous vehicles, natural language
processing, and countless other domains.
"""


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# B.1 — Usa Gemini para extraer los 3 tipos de machine learning mencionados

model = genai.GenerativeModel("gemini-2.5-flash")

# Tu prompt aquí — pide a Gemini que extraiga los 3 tipos de ML
# El resultado debe ser una lista de strings: ["supervised", "unsupervised", "reinforcement"]

response = model.generate_content(
    f"Given the following text, identify and list the three main types of machine learning mentioned. Your response should be a Python list containing only the names of these types in lowercase, e.g., ['type1', 'type2', 'type3'].\n\nText: {texto_largo}"
)

tipos_ml_raw = response.text.strip()

try:
    # Attempt to parse the response as a Python list
    tipos_ml = eval(tipos_ml_raw)
    # Ensure the elements are strings and clean them up
    if isinstance(tipos_ml, list):
        tipos_ml = [str(item).lower().strip() for item in tipos_ml]
    else:
        tipos_ml = None # If not a list, set to None for error handling
except (SyntaxError, NameError): # Handle cases where eval fails
    tipos_ml = None

# Fallback if parsing fails or returns unexpected format
if not isinstance(tipos_ml, list) or len(tipos_ml) != 3:
    # Try to extract them manually if parsing as list failed
    if "supervised learning" in tipos_ml_raw.lower() and \
       "unsupervised learning" in tipos_ml_raw.lower() and \
       "reinforcement learning" in tipos_ml_raw.lower():
        tipos_ml = ["supervised", "unsupervised", "reinforcement"]
    else:
        tipos_ml = None # If still can't find them, set to None

print(f"Tipos de ML: {tipos_ml}")

Tipos de ML: ['supervised learning', 'unsupervised learning', 'reinforcement learning']


In [ ]:
# Using the available model: gemini-2.5-flash
model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(
    f"Given the following text, identify and list the three main types of machine learning mentioned. Your response should be a Python list containing only the names of these types in lowercase, e.g., ['type1', 'type2', 'type3'].\n\nText: {texto_largo}"
)

tipos_ml_raw = response.text.strip()

try:
    # Attempt to parse the response as a Python list
    tipos_ml = eval(tipos_ml_raw)
    # Ensure the elements are strings and clean them up
    if isinstance(tipos_ml, list):
        tipos_ml = [str(item).lower().strip() for item in tipos_ml]
    else:
        tipos_ml = None # If not a list, set to None for error handling
except (SyntaxError, NameError): # Handle cases where eval fails
    tipos_ml = None

# Fallback if parsing fails or returns unexpected format
if not isinstance(tipos_ml, list) or len(tipos_ml) != 3:
    # Try to extract them manually if parsing as list failed
    if "supervised learning" in tipos_ml_raw.lower() and \
       "unsupervised learning" in tipos_ml_raw.lower() and \
       "reinforcement learning" in tipos_ml_raw.lower():
        tipos_ml = ["supervised", "unsupervised", "reinforcement"]
    else:
        tipos_ml = None # If still can't find them, set to None

print(f"Tipos de ML: {tipos_ml}")

Tipos de ML: ['supervised learning', 'unsupervised learning', 'reinforcement learning']


Let's check the available models to understand why the `NotFound` error is occurring. This will list all models accessible through the API and their supported methods.

In [ ]:
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [ ]:
# Validación B.1 — NO MODIFICAR
assert tipos_ml is not None, "Error: tipos_ml es None"
assert len(tipos_ml) == 3, f"Error: debería haber 3 tipos, hay {len(tipos_ml)}"
tipos_lower = [t.lower().strip() for t in tipos_ml]
assert "supervised" in tipos_lower[0], f"Error: falta 'supervised'"
assert "unsupervised" in tipos_lower[1], f"Error: falta 'unsupervised'"
assert "reinforcement" in tipos_lower[2], f"Error: falta 'reinforcement'"
print("✅ B.1 — Extracción con Gemini: CORRECTO")

✅ B.1 — Extracción con Gemini: CORRECTO


### B.2 — Chat multi-turno (15 pts)

Crea un chat con Gemini que:
1. Le preguntes "¿Qué es Python?" → Guarda la respuesta.
2. Le preguntes "¿Y cuáles son sus principales ventajas?" → Guarda la respuesta.
3. Le preguntes "Resume todo lo anterior en una frase" → Guarda la respuesta.

Verifica que el chat mantiene el contexto.

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash")
chat = model.start_chat(history=[])

# Tu código aquí — haz las 3 preguntas y guarda cada respuesta

respuesta_1_obj = chat.send_message("¿Qué es Python?")
respuesta_1 = respuesta_1_obj.text  # Respuesta a "¿Qué es Python?"

respuesta_2_obj = chat.send_message("¿Y cuáles son sus principales ventajas?")
respuesta_2 = respuesta_2_obj.text  # Respuesta a "¿Y cuáles son sus principales ventajas?"

respuesta_3_obj = chat.send_message("Resume todo lo anterior en una frase")
respuesta_3 = respuesta_3_obj.text  # Respuesta a "Resume todo lo anterior en una frase"

num_mensajes_historial = len(chat.history)  # len(chat.history)

print(f"Mensajes en historial: {num_mensajes_historial}")

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 989.32ms


Mensajes en historial: 6


In [ ]:
# Validación B.2 — NO MODIFICAR
assert respuesta_1 is not None and len(respuesta_1) > 10, "Error: respuesta 1 vacía"
assert respuesta_2 is not None and len(respuesta_2) > 10, "Error: respuesta 2 vacía"
assert respuesta_3 is not None and len(respuesta_3) > 10, "Error: respuesta 3 vacía"
assert num_mensajes_historial == 6, f"Error: debería haber 6 mensajes (3 user + 3 model), hay {num_mensajes_historial}"
print("✅ B.2 — Chat multi-turno: CORRECTO")

✅ B.2 — Chat multi-turno: CORRECTO


---
## Parte C — Mini RAG (35 puntos)

Construye un sistema RAG sencillo que responda preguntas sobre un corpus dado.

In [ ]:
!pip uninstall -y langchain langchain-google-genai langchain-community chromadb
!pip install langchain langchain-google-genai langchain-community chromadb -U

Found existing installation: langchain 1.3.0
Uninstalling langchain-1.3.0:
  Successfully uninstalled langchain-1.3.0
Found existing installation: langchain-google-genai 4.2.2
Uninstalling langchain-google-genai-4.2.2:
  Successfully uninstalled langchain-google-genai-4.2.2
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
  Using cached langchain-1.3.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_google_genai-4.2.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
Using cached langchain-1.3.0-py3-none-any.whl (114 kB)
Using cached langchain_google_genai-4.2.2-py3-none-any.whl (67 kB)
Using cached l

In [ ]:
# Corpus de documentos — NO MODIFICAR
corpus = [
    "Python fue creado por Guido van Rossum y lanzado en 1991. Es un lenguaje de programación de alto nivel, interpretado y de propósito general.",
    "Python destaca por su sintaxis limpia y legible. Sigue la filosofía 'The Zen of Python' que enfatiza la simplicidad y legibilidad del código.",
    "Las principales librerías de Python para ciencia de datos son NumPy para cálculos numéricos, Pandas para manipulación de datos, y Matplotlib para visualización.",
    "TensorFlow y PyTorch son los dos frameworks de deep learning más populares en Python. Keras es una API de alto nivel que funciona sobre TensorFlow.",
    "Django y Flask son los frameworks web más usados en Python. Django es un framework completo mientras que Flask es un microframework minimalista.",
    "Python 3.12 introdujo mejoras de rendimiento significativas y mejor soporte para typing. Python 2 dejó de recibir soporte en enero de 2020.",
    "El gestor de paquetes oficial de Python es pip, y los entornos virtuales se crean con venv o conda. PyPI es el repositorio oficial de paquetes.",
    "Python se utiliza ampliamente en inteligencia artificial, machine learning, automatización, desarrollo web, ciencia de datos y scripting."
]

In [ ]:
# C — Construye tu sistema RAG
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

# Tu código aquí:
# 1. Crea embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# 2. Crea base vectorial con Chroma
vectorstore = Chroma.from_texts(corpus, embeddings)

# 3. Configura el LLM (Gemini)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

# 4. Crea cadena RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

ModuleNotFoundError: No module named 'langchain.chains'

### C.1 — Pregunta: ¿Quién creó Python? (10 pts)

In [ ]:
# C.1 — Responde usando tu RAG
respuesta_creador = qa_chain.invoke("¿Quién creó Python?")['result']  # El resultado de tu qa_chain

print(f"Respuesta: {respuesta_creador}")

NameError: name 'qa_chain' is not defined

In [ ]:
# Validación C.1 — NO MODIFICAR
assert respuesta_creador is not None, "Error: respuesta vacía"
assert "guido" in respuesta_creador.lower(), f"Error: la respuesta debe mencionar a Guido van Rossum"
print("✅ C.1 — Creador de Python: CORRECTO")

### C.2 — Pregunta: ¿Cuáles son los frameworks de deep learning? (10 pts)

In [ ]:
# C.2 — Pregunta sobre frameworks de deep learning
respuesta_frameworks = qa_chain.invoke("¿Cuáles son los frameworks de deep learning?")['result']  # El resultado de tu qa_chain

print(f"Respuesta: {respuesta_frameworks}")

NameError: name 'qa_chain' is not defined

In [ ]:
# Validación C.2 — NO MODIFICAR
assert respuesta_frameworks is not None, "Error: respuesta vacía"
resp_lower = respuesta_frameworks.lower()
assert "tensorflow" in resp_lower or "pytorch" in resp_lower, \
    "Error: la respuesta debe mencionar TensorFlow o PyTorch"
print("✅ C.2 — Frameworks DL: CORRECTO")

### C.3 — Pregunta: ¿Para qué se usa Python? (15 pts)

In [ ]:
# C.3 — Pregunta sobre usos de Python
respuesta_usos = qa_chain.invoke("¿Para qué se usa Python?")['result']  # El resultado de tu qa_chain

print(f"Respuesta: {respuesta_usos}")

NameError: name 'qa_chain' is not defined

In [ ]:
# Validación C.3 — NO MODIFICAR
assert respuesta_usos is not None, "Error: respuesta vacía"
resp_lower = respuesta_usos.lower()
usos_mencionados = sum(1 for uso in ["inteligencia artificial", "machine learning", "web", "datos", "automatización"]
                       if uso in resp_lower or uso.split()[0] in resp_lower)
assert usos_mencionados >= 2, f"Error: la respuesta debe mencionar al menos 2 usos de Python"
print("✅ C.3 — Usos de Python: CORRECTO")

---
## 📋 Autoevaluación — Responde en Blackboard

Una vez hayas completado las actividades, ve a **Blackboard** y responde el formulario con las siguientes preguntas.

---

### Pregunta 1 (Verdadero / Falso)

**La función `pipeline()` de HuggingFace permite usar modelos pre-entrenados con muy pocas líneas de código, sin necesidad de entrenar nada.**

---

### Pregunta 2 (Multirespuesta)

**¿Qué tipo de pipeline se usa para clasificar texto en categorías sin haberlo entrenado específicamente para esas categorías?**

- a) `sentiment-analysis`
- b) `text-generation`
- c) `zero-shot-classification`
- d) `question-answering`

---

### Pregunta 3 (Verdadero / Falso)

**De las 10 reseñas del ejercicio (Parte A.1), hay exactamente 6 positivas y 4 negativas según el pipeline de sentiment-analysis.**

---

### Pregunta 4 (Multirespuesta)

**En un sistema RAG, ¿cuál es la función del paso de "retrieval" (recuperación)?**

- a) Generar la respuesta final con el LLM
- b) Entrenar el modelo con los documentos
- c) Buscar los documentos más relevantes de la base de conocimiento
- d) Tokenizar el texto de entrada

---

### Pregunta 5 (Verdadero / Falso)

**La API de chat de Gemini mantiene el contexto de la conversación a lo largo de múltiples mensajes (chat multi-turno).**

---

### Pregunta 6 (Multirespuesta)

**En un sistema RAG, ¿qué componente almacena los embeddings vectoriales para la búsqueda por similitud?**

- a) El LLM (Large Language Model)
- b) El tokenizer
- c) La base de datos vectorial (ej: Chroma)
- d) El prompt template

---

### Pregunta 7 (Verdadero / Falso)

**Zero-shot classification requiere hacer fine-tuning del modelo con nuestras categorías específicas antes de poder usarlo.**

---

### Pregunta 8 (Multirespuesta)

**Si hacemos 3 preguntas en un chat multi-turno con Gemini (Parte B.2), ¿cuántos mensajes tiene el historial del chat?**

- a) 3
- b) 4
- c) 6
- d) 9